In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text, bindparam
import os
import numpy as np
import re
from dotenv import load_dotenv

#### Database Connection Setup
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

DB = DB_URL
engine = create_engine(DB, pool_pre_ping=True)



### IDENTIFY CPT CODES FOR QUERIES DRIVEN BY CPT CODES CSV FILE
curr_path = os.getcwd()
cpts = pd.read_csv(f"{curr_path}/dimensions/cpt codes mapping.csv")


cpt_list = cpts['cpt_code'].astype(str).str.strip().tolist()
# if your CPT codes are numeric in the DB, you can convert them to int:
# cpt_list = [int(x) if x.isdigit() else x for x in cpt_list]

if not cpt_list:
	raise ValueError("cpt_list is empty; check the CSV path and the 'cpt_code' column.")

cpts['cpt_code'] = cpts['cpt_code'].astype(str)

### SQL QUERY

sql = text("""
select b.hospital_name, b.hospital_address, 
		   hcc.description, hcc.code, hcc.setting, hcc.modifiers, hcc.standard_charge_gross, hcc.standard_charge_discounted_cash, hcc.payer_name, hcc.plan_name, 
		   hcc.standard_charge_negotiated_dollar, standard_charge_negotiated_percentage, hcc.estimated_amount, hcc.standard_charge_min, hcc.standard_charge_max		   
from hospital_cpt_charges hcc
join hospital_metadata b
on hcc.source_file=b.source_file
WHERE hcc.code IN :cpts
""").bindparams(bindparam("cpts", expanding=True))


df = pd.read_sql_query(sql, engine, params={"cpts": cpt_list})

merge_df = pd.merge(df, cpts, left_on='code', right_on='cpt_code', how='left')


### Update Charge Columns to Numeric
cols_to_float = [
    'standard_charge_gross',
    'standard_charge_negotiated_dollar',
    'standard_charge_negotiated_percentage',
    'estimated_amount',
    'standard_charge_min',
    'standard_charge_max'
]

# Clean and convert to float safely
for col in cols_to_float:
    merge_df[col] = (
        merge_df[col]
        .astype(str)                           # ensure string type
        .str.replace(',', '', regex=False)     # remove commas like "1,000"
        .str.replace('$', '', regex=False)     # remove dollar signs if any
        .replace(['', 'None', 'nan', 'NaN'], np.nan)  # treat blanks as NaN
        .astype(float)
    )


#### Determine Rate Amount of the Procedures
SENTINEL = 999999999.0

pct   = merge_df['standard_charge_negotiated_percentage']
gross = merge_df['standard_charge_gross']
doll  = merge_df['standard_charge_negotiated_dollar']
est   = merge_df['estimated_amount']
smax  = merge_df['standard_charge_max']

# If your percentages are like 55.0 for 55%, keep /100. If already 0.55, remove /100.
pct_factor = pct / 100.0

# Masks
has_dollar      = doll.notna()
has_pct_gross   = pct.notna() & gross.notna()
has_pct_est     = pct.notna() & est.notna() & (est != 0) & (est != SENTINEL)
has_pct_max     = pct.notna() & smax.notna()

no_negotiated   = doll.isna() & pct.isna()
fallback_est    = no_negotiated & est.notna() & (est != 0) & (est != SENTINEL)
fallback_gross  = no_negotiated & gross.notna()
fallback_max    = no_negotiated & smax.notna()

merge_df['Rate'] = np.select(
    [
        # 1) negotiated dollar
        has_dollar,

        # 2) percentage path (priority: gross -> estimated -> max)
        has_pct_gross,
        has_pct_est,
        has_pct_max,

        # 3) when BOTH negotiated fields are null → fallbacks
        fallback_est,
        fallback_gross,
        fallback_max,
    ],
    [
        doll,
        gross * pct_factor,
        est   * pct_factor,
        smax  * pct_factor,
        est,
        gross,
        smax,
    ],
    default=np.nan
)

merge_df['hospital_address'] = merge_df['hospital_address'].str.replace(
    r'(\b\d{5})(\d{4}\b)',
    r'\1-\2',
    regex=True
)

In [ ]:
import pandas as pd
import numpy as np
import re

# 1) make a list of addresses per row (split on | with or without spaces) ---
merge_df = merge_df.copy()

merge_df['address_list'] = (
    merge_df['hospital_address']
      .fillna('')
      .apply(lambda x: [a.strip() for a in re.split(r'\s*\|\s*', str(x)) if a.strip()])
)

# how many locations were in that row
merge_df['num_locations'] = merge_df['address_list'].apply(len)

# --- 2) explode to one row per address ---
exploded = (
    merge_df
      .explode('address_list', ignore_index=True)
      .rename(columns={'address_list': 'hospital_address_single'})
)

# flag whether the price was system-applied (multiple addresses) or facility-specific
exploded['price_scope'] = np.where(exploded['num_locations'] > 1, 'system_applied', 'facility_specific')

# --- 3) improved ZIP extraction (take LAST ZIP, not first) ---
# find all 5-digit or ZIP+4 patterns
zip_candidates = exploded['hospital_address_single'].str.findall(r'\b\d{5}(?:-\d{4})?\b')

# ZIP4 = last candidate in each address, if any exist
exploded['ZIP4'] = zip_candidates.apply(lambda xs: xs[-1] if isinstance(xs, list) and len(xs) else pd.NA)

# ZIP = first 5 digits of ZIP4
exploded['ZIP'] = exploded['ZIP4'].astype('string').str.slice(0, 5)

# ensure text type (important for leading zeros)
exploded['ZIP']  = exploded['ZIP'].astype('string')
exploded['ZIP4'] = exploded['ZIP4'].astype('string')

# --- 4) (optional) tidy columns/order ---
cols_front = ['hospital_name', 'hospital_address_single', 'ZIP', 'ZIP4', 'num_locations', 'price_scope']
other_cols = [c for c in exploded.columns if c not in cols_front]
exploded = exploded[cols_front + other_cols]


In [3]:
check = exploded[exploded['hospital_name'] == "GULF COAST MEDICAL CENTER"]
display(check.head(10))

,hospital_name,hospital_address_single,ZIP,ZIP4,num_locations,price_scope,hospital_address,description,code,setting,...,plan_name,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,estimated_amount,standard_charge_min,standard_charge_max,cpt_code,Description,Specialty,Rate
620,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",Drain/Inject Small Joint/Bursa,20600,outpatient,...,AETNA PPO [21010105],1458.5,NaN,6786.53,7405.44,14810.88,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,1458.5000
684,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",HC IR Drain/Inject Sm Jnt/Bursa,20600,outpatient,...,AETNA PPO [21010105],NaN,62.30,588.74,378.00,756.00,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,588.7350
685,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",HC Evl Drain/Inject Sm Jnt/Bursa,20600,outpatient,...,AETNA PPO [21010105],NaN,62.30,0.62,0.40,0.80,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,0.6230
686,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",Drain/Inject Small Joint/Bursa,20600,outpatient,...,AETNA PPO [21010105],NaN,62.30,130.83,84.00,168.00,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,130.8300
1031,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",Drain/Inject Small Joint/Bursa,20600,outpatient,...,AVMED HEALTH PLAN CONTRACTED [25020401],NaN,59.00,10923.05,7405.44,14810.88,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,10923.0240
1097,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",HC IR Drain/Inject Sm Jnt/Bursa,20600,outpatient,...,AVMED HEALTH PLAN CONTRACTED [25020401],NaN,59.00,557.55,378.00,756.00,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,557.5500
1102,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",HC Evl Drain/Inject Sm Jnt/Bursa,20600,outpatient,...,AVMED HEALTH PLAN CONTRACTED [25020401],NaN,59.00,0.59,0.40,0.80,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,0.5900
1116,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",Drain/Inject Small Joint/Bursa,20600,outpatient,...,AVMED HEALTH PLAN CONTRACTED [25020401],NaN,59.00,123.90,84.00,168.00,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,123.9000
1828,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",Drain/Inject Small Joint/Bursa,20600,outpatient,...,BC FL PPO [21000101],5250.0,NaN,5250.00,7405.44,14810.88,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,5250.0000
2138,GULF COAST MEDICAL CENTER,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",33912,33912-4300,1,facility_specific,"13681 DOCTORS WAY, FORT MYERS, FL 33912-4300",HC IR Drain/Inject Sm Jnt/Bursa,20600,outpatient,...,BC FL PPO [21000101],NaN,69.27,945.00,378.00,756.00,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,654.6015


In [4]:
print(exploded.columns)
cols_keep = [
    'hospital_name', 'hospital_address_single', 'ZIP4', 'setting', 'modifiers', 'standard_charge_gross',
    'standard_charge_discounted_cash', 'payer_name', 'plan_name',
    'standard_charge_negotiated_dollar', 'standard_charge_negotiated_percentage', 'estimated_amount',
    'standard_charge_min', 'standard_charge_max', 'cpt_code', 'Description',
    'Specialty', 'Rate'
]

missing = [c for c in cols_keep if c not in exploded.columns]

exploded = exploded[cols_keep].copy()

Index(['hospital_name', 'hospital_address_single', 'ZIP', 'ZIP4',
       'num_locations', 'price_scope', 'hospital_address', 'description',
       'code', 'setting', 'modifiers', 'standard_charge_gross',
       'standard_charge_discounted_cash', 'payer_name', 'plan_name',
       'standard_charge_negotiated_dollar',
       'standard_charge_negotiated_percentage', 'estimated_amount',
       'standard_charge_min', 'standard_charge_max', 'cpt_code', 'Description',
       'Specialty', 'Rate'],
      dtype='object')


In [5]:
display(exploded.head(10))

,hospital_name,hospital_address_single,ZIP4,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,payer_name,plan_name,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,estimated_amount,standard_charge_min,standard_charge_max,cpt_code,Description,Specialty,Rate
0,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,penfield,care management inc penfieldcaremanagementinc,NaN,55.0,999999999.0,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,5053.40
1,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,cigna,cignahealthplansurefit,1045.00,NaN,NaN,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,1045.00
2,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,avmed,avmedemployeroptionaso,9188.00,NaN,NaN,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,9188.00
3,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,us,and uk medical abroad llc ppo usandukmedicalab...,NaN,55.0,999999999.0,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,5053.40
4,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,beacon,health options beaconhealthoptionsmgdmcare,NaN,50.0,999999999.0,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,4594.00
5,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,bcbs,-fl bcbsflppc,2265.00,NaN,NaN,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,2265.00
6,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,magellan,magellanbehavioral,NaN,60.0,999999999.0,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,5512.80
7,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,tour,plus med tourplusmedassistanceppo,69.65,NaN,NaN,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,69.65
8,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,preferred,governmental preferredgovernmentalclaimsolution,5609.50,NaN,NaN,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,5609.50
9,St. Marys Medical Center - West Palm Beach,"901 45th St, West Palm Beach, FL 33407",33407,outpatient,None,NaN,None,axis,services inc axisservicesinc,NaN,60.0,999999999.0,0.95,9188.0,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,5512.80


In [10]:
exploded.to_csv(f"{curr_path}/exploded_cpt_data.csv", index=False)

In [7]:
# # filtered = exploded[(exploded['hospital_name'] == "AdventHealth North Pinellas") & (exploded["code"]=="20600") & (exploded['payer_name']=="blue_cross_&_blue_shield_of_florida")]
# filtered = exploded[(exploded['hospital_address_single'] == "10080 Lake Nona Boulevard, Orlando, FL 32827")]

# display(filtered.head(15))

In [8]:
# count rows where Rate is blank (NaN)
missing_count = merge_df['Rate'].isna().sum()
total_rows = len(merge_df)
print(f"Missing Rate count: {missing_count} / {total_rows} ({missing_count/total_rows:.2%})")

# optional: keep the rows for inspection
missing_rate_rows = merge_df[merge_df['Rate'].isna()]

Missing Rate count: 9478 / 1659687 (0.57%)


In [9]:
# Display rows where Rate is null (uses existing `missing_rate_rows` and `merge_df`)
print(f"Missing Rate rows: {missing_rate_rows.shape[0]} / {len(merge_df)}")
missing_rate_rows.head(200)

Missing Rate rows: 9478 / 1659687


,hospital_name,hospital_address,description,code,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,payer_name,plan_name,...,standard_charge_negotiated_percentage,estimated_amount,standard_charge_min,standard_charge_max,cpt_code,Description,Specialty,Rate,address_list,num_locations
1587,"Bay County Health System, LLC",615 N. Bonita Ave Panama City FL 32401,DRAIN/INJ JOINT/BURSA W/O US,20600,OUTPATIENT,None,NaN,None,AETNA MCR REPLACEMENT,2284_AETNA MEDICARE REPLACEMENT OUTPATIENT BMF...,...,103.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[615 N. Bonita Ave Panama City FL 32401],1
1588,"Bay County Health System, LLC",615 N. Bonita Ave Panama City FL 32401,DRAIN/INJ JOINT/BURSA W/O US,20600,INPATIENT,None,NaN,None,AETNA MCR REPLACEMENT,2391_AETNA MEDICARE REPLACEMENT INPATIENT BMFL...,...,103.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[615 N. Bonita Ave Panama City FL 32401],1
1589,"Bay County Health System, LLC",615 N. Bonita Ave Panama City FL 32401,DRAIN/INJ JOINT/BURSA W/O US,20600,OUTPATIENT,None,NaN,None,AMBETTER COMMERCIAL-EXCHANGE,2245_SUNSHINE HEALTH AMBETTER COMMERCIAL OUTPA...,...,165.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[615 N. Bonita Ave Panama City FL 32401],1
1590,"Bay County Health System, LLC",615 N. Bonita Ave Panama City FL 32401,DRAIN/INJ JOINT/BURSA W/O US,20600,INPATIENT,None,NaN,None,AMBETTER COMMERCIAL-EXCHANGE,2385_SUNSHINE HEALTH AMBETTER COMMERCIAL INPAT...,...,165.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[615 N. Bonita Ave Panama City FL 32401],1
1591,"Bay County Health System, LLC",615 N. Bonita Ave Panama City FL 32401,DRAIN/INJ JOINT/BURSA W/O US,20600,OUTPATIENT,None,NaN,None,BLUE MCR REPLACEMENT,2260_MEDICARE ADVANTAGE BLUE OUTPATIENT BMFL 2...,...,100.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[615 N. Bonita Ave Panama City FL 32401],1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66370,"St. Luke's-St. Vincent's Healthcare, Inc.",4201 Belfort Rd Jacksonville FL 32216,DRAIN/INJ JOINT/BURSA W/O US,20600,INPATIENT,None,NaN,None,AMBETTER COMMERCIAL-EXCHANGE,1346_SUNSHINE AMBETTER EXCHANGE COMMERCIAL INP...,...,190.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[4201 Belfort Rd Jacksonville FL 32216],1
66371,"St. Luke's-St. Vincent's Healthcare, Inc.",4201 Belfort Rd Jacksonville FL 32216,DRAIN/INJ JOINT/BURSA W/O US,20600,OUTPATIENT,None,NaN,None,BC ADVANTAGE MCR REPLACEMENT,1267_MEDICARE ADVANTAGE BLUE CROSS OUTPATIENT ...,...,100.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[4201 Belfort Rd Jacksonville FL 32216],1
66372,"St. Luke's-St. Vincent's Healthcare, Inc.",4201 Belfort Rd Jacksonville FL 32216,DRAIN/INJ JOINT/BURSA W/O US,20600,INPATIENT,None,NaN,None,BC ADVANTAGE MCR REPLACEMENT,1351_MEDICARE ADVANTAGE BLUE CROSS INPATIENT 2...,...,100.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[4201 Belfort Rd Jacksonville FL 32216],1
66373,"St. Luke's-St. Vincent's Healthcare, Inc.",4201 Belfort Rd Jacksonville FL 32216,DRAIN/INJ JOINT/BURSA W/O US,20600,OUTPATIENT,None,NaN,None,BLUE CROSS ALIGNMENT,1268_MEDICARE ADVANTAGE ALIGNMENT HEALTHCARE O...,...,100.0,999999999.0,NaN,NaN,20600,"Arthrocentesis, aspiration and/or injection, s...",Orthopedic,NaN,[4201 Belfort Rd Jacksonville FL 32216],1
